# Modal Component

> Tabbed modal with progress, logs, and resources tabs.

In [ ]:
#| default_exp components.modal

In [ ]:
#| export
from typing import Optional

from fasthtml.common import Div, H3, Button, Input, Form, Span, Dialog, FT

from cjm_fasthtml_daisyui.components.actions.modal import modal, modal_box, modal_action, modal_backdrop
from cjm_fasthtml_daisyui.components.actions.button import (
    btn, btn_colors, btn_sizes, btn_styles, btn_modifiers,
)
from cjm_fasthtml_daisyui.components.navigation.tabs import tab, tabs, tabs_styles, tab_content
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui, border_dui, text_dui
from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.sizing import w, max_w
from cjm_fasthtml_tailwind.utilities.typography import font_size, font_weight
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import flex_display, items, justify
from cjm_fasthtml_tailwind.utilities.layout import display_tw, position, right, top
from cjm_fasthtml_tailwind.core.base import combine_classes

from cjm_fasthtml_job_monitor.html_ids import JobMonitorHtmlIds
from cjm_fasthtml_job_monitor.models import JobMonitorUrls, JobMonitorConfig, ResourceSnapshot
from cjm_fasthtml_job_monitor.components.tabs.progress_tab import render_progress_tab
from cjm_fasthtml_job_monitor.components.tabs.logs_tab import render_logs_tab
from cjm_fasthtml_job_monitor.components.tabs.resources_tab import render_resources_tab

## Poll Anchor

The poll anchor is a hidden div that carries the HTMX polling attributes.
It targets itself for replacement. On each poll cycle, the progress route
returns an updated poll anchor (to continue or stop polling) plus OOB swaps
for each tab panel's inner content. This keeps the tab radio inputs untouched.

In [ ]:
#| export
_TERMINAL_STATUSES = {'completed', 'failed', 'cancelled'}

def render_poll_anchor(
    ids: JobMonitorHtmlIds,       # Element IDs
    urls: JobMonitorUrls,         # Route URLs
    config: JobMonitorConfig,     # Display config
    is_active: bool = True,       # Whether polling should continue
) -> FT:  # Hidden poll anchor div
    """Render the hidden polling anchor element.
    
    When `is_active`, includes hx-get/hx-trigger for continued polling.
    When not active, renders as an inert hidden div (stops polling).
    """
    attrs = {"id": ids.poll_anchor, "style": "display:none;"}
    if is_active:
        attrs["hx_get"] = urls.progress
        attrs["hx_trigger"] = f"every {config.poll_interval_ms}ms"
        attrs["hx_swap"] = "outerHTML"
    return Div(**attrs)


def render_tab_content_oob(
    ids: JobMonitorHtmlIds,                      # Element IDs
    status: str = 'pending',                     # Job status
    progress_value: float = 0.0,                 # 0.0 to 1.0
    status_message: str = '',                    # Stage message
    started_at: Optional[float] = None,          # Unix timestamp
    completed_at: Optional[float] = None,        # Unix timestamp
    logs: str = '',                              # Log text
    resources: Optional[ResourceSnapshot] = None, # Resource data
) -> tuple:  # (progress_div, logs_div, resources_div) with OOB attrs
    """Render the three tab inner content divs as OOB swap targets."""
    progress_div = Div(
        render_progress_tab(
            ids, status=status, progress_value=progress_value,
            status_message=status_message, started_at=started_at,
            completed_at=completed_at,
        ),
        id=ids.tab_progress,
    )
    progress_div.attrs['hx-swap-oob'] = "true"

    logs_div = Div(
        render_logs_tab(ids, logs=logs),
        id=ids.tab_logs,
    )
    logs_div.attrs['hx-swap-oob'] = "true"

    resources_div = Div(
        render_resources_tab(resources),
        id=ids.tab_resources,
    )
    resources_div.attrs['hx-swap-oob'] = "true"

    return progress_div, logs_div, resources_div


def render_footer_oob(
    ids: JobMonitorHtmlIds,    # Element IDs
    urls: JobMonitorUrls,      # Route URLs
    is_active: bool = True,    # Whether job is active (show cancel)
) -> FT:  # Footer div with OOB attr
    """Render the modal footer with cancel button as OOB swap."""
    children = []
    if is_active:
        children.append(
            Button(
                "Cancel",
                hx_post=urls.cancel,
                hx_swap="none",
                cls=combine_classes(btn, btn_colors.error, btn_styles.outline, btn_sizes.sm),
            )
        )
    footer = Div(
        *children,
        id=ids.modal_footer,
        cls=combine_classes(flex_display, justify.end, m.t(4)),
    )
    footer.attrs['hx-swap-oob'] = "true"
    return footer


def render_poll_response(
    config: JobMonitorConfig,                    # Display config
    ids: JobMonitorHtmlIds,                      # Element IDs
    urls: JobMonitorUrls,                        # Route URLs
    status: str = 'pending',                     # Job status
    progress_value: float = 0.0,                 # 0.0 to 1.0
    status_message: str = '',                    # Stage message
    started_at: Optional[float] = None,          # Unix timestamp
    completed_at: Optional[float] = None,        # Unix timestamp
    logs: str = '',                              # Log text
    resources: Optional[ResourceSnapshot] = None, # Resource data
) -> tuple:  # (poll_anchor, progress_oob, logs_oob, resources_oob, footer_oob)
    """Render the poll response: updated poll anchor + OOB tab content updates.
    
    This is the primary response for the progress polling route.
    The poll anchor is the 'primary' swap target (outerHTML on itself).
    The three tab content divs and footer are OOB swaps.
    """
    is_active = status not in _TERMINAL_STATUSES

    anchor = render_poll_anchor(ids, urls, config, is_active=is_active)
    progress_div, logs_div, resources_div = render_tab_content_oob(
        ids, status=status, progress_value=progress_value,
        status_message=status_message, started_at=started_at,
        completed_at=completed_at, logs=logs, resources=resources,
    )
    footer = render_footer_oob(ids, urls, is_active=is_active)

    return (anchor, progress_div, logs_div, resources_div, footer)

## render_job_modal

Full modal dialog with header, static tab structure, inner content divs,
poll anchor, and backdrop. The tab structure is rendered once and never
replaced. Only the inner content of each tab and the footer are updated
via OOB swaps from the progress route.

In [ ]:
#| export
def render_job_modal(
    config: JobMonitorConfig,                    # Display config
    ids: JobMonitorHtmlIds,                      # Element IDs
    urls: JobMonitorUrls,                        # Route URLs
    status: str = 'pending',                     # Job status
    progress_value: float = 0.0,                 # 0.0 to 1.0
    status_message: str = '',                    # Stage message
    started_at: Optional[float] = None,          # Unix timestamp
    completed_at: Optional[float] = None,        # Unix timestamp
    logs: str = '',                              # Log text
    resources: Optional[ResourceSnapshot] = None, # Resource data
    open_on_render: bool = False,                # Auto-open via JS
) -> FT:  # Dialog element
    """Render the full tabbed modal dialog.
    
    The tab structure (radio inputs + tab-content wrappers) is static.
    Each tab-content wrapper contains an inner div with a stable ID
    that gets OOB-swapped by the progress route. This prevents the
    selected tab from resetting on each poll cycle.
    
    Closable via: Escape key, X button (top-right), or clicking backdrop.
    """
    is_active = status not in _TERMINAL_STATUSES
    tab_group = f"{ids.prefix}-modal-tabs"

    # Initial tab inner content
    progress_inner = render_progress_tab(
        ids, status=status, progress_value=progress_value,
        status_message=status_message, started_at=started_at,
        completed_at=completed_at,
    )
    logs_inner = render_logs_tab(ids, logs=logs)
    resources_inner = render_resources_tab(resources)

    # Tab structure — radio inputs and tab-content wrappers are static
    tab_children = [
        # Progress tab
        Input(type="radio", name=tab_group, aria_label="Progress",
              checked="checked", cls=str(tab)),
        Div(
            Div(progress_inner, id=ids.tab_progress),
            cls=combine_classes(tab_content, border_dui.base_300, bg_dui.base_100),
        ),
        # Logs tab
        Input(type="radio", name=tab_group, aria_label="Logs", cls=str(tab)),
        Div(
            Div(logs_inner, id=ids.tab_logs),
            cls=combine_classes(tab_content, border_dui.base_300, bg_dui.base_100),
        ),
        # Resources tab
        Input(type="radio", name=tab_group, aria_label="Resources", cls=str(tab)),
        Div(
            Div(resources_inner, id=ids.tab_resources),
            cls=combine_classes(tab_content, border_dui.base_300, bg_dui.base_100),
        ),
    ]

    tab_container = Div(
        *tab_children,
        cls=combine_classes(tabs, tabs_styles.border, w.full),
    )

    # Footer with cancel button
    footer_children = []
    if is_active:
        footer_children.append(
            Button(
                "Cancel",
                hx_post=urls.cancel,
                hx_swap="none",
                cls=combine_classes(btn, btn_colors.error, btn_styles.outline, btn_sizes.sm),
            )
        )
    footer = Div(
        *footer_children,
        id=ids.modal_footer,
        cls=combine_classes(flex_display, justify.end, m.t(4)),
    )

    # Poll anchor (hidden, drives HTMX polling)
    poll = render_poll_anchor(ids, urls, config, is_active=is_active)

    # X close button (top-right corner)
    close_btn = Form(
        Button(
            "\u2715",  # Unicode multiplication sign (✕)
            cls=combine_classes(
                btn, btn_sizes.sm, btn_modifiers.circle, btn_styles.ghost,
                position.absolute, right._2, top._2,
            ),
        ),
        method="dialog",
    )

    # Open script
    from fasthtml.common import Script
    open_script = Script(
        f"document.getElementById('{ids.modal}').showModal();"
    ) if open_on_render else Span()

    return Dialog(
        Div(
            # X close button
            close_btn,
            # Header
            H3(config.modal_title, cls=combine_classes(font_weight.bold, font_size.lg)),
            # Tab structure (static)
            tab_container,
            # Footer (OOB-swappable)
            footer,
            # Poll anchor (hidden)
            poll,
            cls=combine_classes(modal_box, max_w._2xl, w.full),
        ),
        # Backdrop — clicking outside closes the modal
        Form(Button("close", cls=combine_classes(btn, btn_styles.ghost)), method="dialog", cls=str(modal_backdrop)),
        open_script,
        id=ids.modal,
        cls=str(modal),
    )

In [ ]:
# Test render_poll_response
from cjm_fasthtml_job_monitor.html_ids import JobMonitorHtmlIds
from cjm_fasthtml_job_monitor.models import JobMonitorUrls, JobMonitorConfig

ids = JobMonitorHtmlIds(prefix="jm")
urls = JobMonitorUrls(trigger="/fa/trigger", progress="/fa/progress", cancel="/fa/cancel")
config = JobMonitorConfig(modal_title="Force Alignment")

# Running state -- should have active poll anchor + OOB tab updates
result = render_poll_response(
    config, ids, urls, status='running', progress_value=0.5,
    status_message='Aligning...',
)
assert len(result) == 5  # anchor + 3 tabs + footer
anchor = result[0]
assert anchor.attrs['id'] == 'jm-poll'
assert 'hx-get' in anchor.attrs
assert 'every 1000ms' in anchor.attrs.get('hx-trigger', '')
# OOB tab content
assert result[1].attrs['id'] == 'jm-tab-progress'
assert result[1].attrs.get('hx-swap-oob') == 'true'
assert result[2].attrs['id'] == 'jm-tab-logs'
assert result[3].attrs['id'] == 'jm-tab-resources'
# Footer
assert result[4].attrs['id'] == 'jm-modal-footer'
print("render_poll_response (running): OK")

# Completed state -- poll anchor should be inert (no hx-get)
result_done = render_poll_response(
    config, ids, urls, status='completed', progress_value=1.0,
)
anchor_done = result_done[0]
assert 'hx-get' not in anchor_done.attrs
print("render_poll_response (completed): poll stopped")

In [ ]:
# Test render_job_modal
modal_el = render_job_modal(
    config, ids, urls, status='running', progress_value=0.3,
    status_message='Loading model...', open_on_render=True,
)
assert modal_el.tag == 'dialog'
assert modal_el.attrs['id'] == 'jm-modal'
assert 'modal' in modal_el.attrs['class']

# Verify static tab structure inside modal_box
from fasthtml.common import to_xml
html = to_xml(modal_el)
# Tab radio inputs should be present
assert 'name="jm-modal-tabs"' in html
assert 'aria-label="Progress"' in html
assert 'aria-label="Logs"' in html
assert 'aria-label="Resources"' in html
# Inner content divs with stable IDs
assert 'id="jm-tab-progress"' in html
assert 'id="jm-tab-logs"' in html
assert 'id="jm-tab-resources"' in html
# Poll anchor
assert 'id="jm-poll"' in html
assert 'hx-get="/fa/progress"' in html
# Footer
assert 'id="jm-modal-footer"' in html
# Open script
assert 'showModal()' in html
print("render_job_modal: OK")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()